In [ ]:
import re
import pandas as pd
import numpy as np
from pathlib import Path
from transformers import Trainer, TrainingArguments, AutoTokenizer, AutoModelForSequenceClassification
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.metrics import confusion_matrix
from tqdm import tqdm
from datasets import Dataset, load_dataset, DatasetDict, Features, ClassLabel, Value
import evaluate
import torch
import os
import json
from datetime import datetime

In [ ]:


def preprocess_data():
    # read in patent data
    g_patent = pd.read_csv('g_patent.tsv', sep='\t', low_memory=False) 
    g_patent = g_patent[g_patent['patent_type'] == 'utility']

    g_assignee = pd.read_csv('g_assignee_disambiguated.tsv', sep='\t', low_memory=False)
    g_assignee = g_assignee[g_assignee['assignee_sequence'] == 0]
    g_assignee = g_assignee[['patent_id', 'assignee_id', 'disambig_assignee_organization', 'location_id']]
    g_patent = g_patent.merge(g_assignee, on='patent_id', how='inner')
    del g_assignee

    g_location = pd.read_csv('g_location_disambiguated.tsv', sep='\t', low_memory=False)
    g_patent = g_patent.merge(g_location[['location_id', 'disambig_country']], on='location_id', how='left')

    pb_assignee = pd.read_excel('matched_result.xlsx')

    pb_marketmap = pd.read_csv('predicted_positive_v2.csv')
    pb_marketmap.sort_values(by=['companyid'], inplace=True)

    pb_marketmap['count_sector'] = pb_marketmap.groupby('companyid')['companyid'].transform('count')
    pb_marketmap['unique_segment'] = pb_marketmap['marketmap'] + pb_marketmap['segment']
    pb_marketmap['count_segment'] = pb_marketmap.groupby('companyid')['unique_segment'].transform('nunique')
    pb_marketmap['count_marketmap'] = pb_marketmap.groupby('companyid')['marketmap'].transform('nunique')

    pb_marketmap = pb_marketmap[pb_marketmap['count_marketmap'] == 1]

    pb_marketmap = pd.merge(pb_marketmap, pb_assignee[['companyid', 'assignee_id']], on='companyid', how='inner')

    df_training = pd.merge(g_patent, pb_marketmap, on='assignee_id', how='inner')
    df_predicting = g_patent[~g_patent['patent_id'].isin(df_training['patent_id'])]
    df_predicting = df_predicting[~df_predicting['disambig_country'].isin(['US', 'CN'])]

    df_predicting['patent_date'] = pd.to_datetime(df_predicting['patent_date'])
    df_predicting = df_predicting[df_predicting['patent_date'].dt.year >= 2000]
    df_predicting = df_predicting.sample(n=100000, random_state=1)

    ls_subsegment = df_training['fullname'].unique()
    ls_subsegment.sort()

    return df_training, df_predicting, ls_subsegment

def subsegment_name_processing(nameStr):
    nameStr = re.sub(r'[^\w\s]|_', '', nameStr)
    return nameStr.replace(" ", "")

def prepare_data_for_subsegment(df_training, SELECTED_SUBSEGMENT):
    df_training = df_training[df_training['patent_abstract'].str.len() > 0].copy()
    df_training.loc[:, 'label'] = 0
    df_training.loc[df_training['fullname'] == SELECTED_SUBSEGMENT, 'label'] = 1

    PARENT_SEGMENT = df_training[df_training['fullname'] == SELECTED_SUBSEGMENT]['segment'].values[0]
    ds_positives = df_training[df_training.label == 1]
    len_positives = len(ds_positives)

    if len_positives < 10:
        print(f'Skip the training for subsegment: {SELECTED_SUBSEGMENT}')
        return None, None

    if len_positives > 1000:
        ds_positives = ds_positives.sample(1000, replace=True)
        len_positives = 1000

    num_sampled = int(len_positives * 0.05)
    ds_negatives_adjacent_subsegments = df_training[(df_training.label == 0) & (df_training.segment == PARENT_SEGMENT)]
    ds_negatives_downsample = df_training[df_training.label == 0].groupby('segment', group_keys=False).apply(lambda x: x.sample(num_sampled, replace=True)) 
    
    if len(ds_negatives_adjacent_subsegments) > len(ds_negatives_downsample):
        ds_negatives_adjacent_subsegments = ds_negatives_adjacent_subsegments.sample(len(ds_negatives_downsample), replace=True)
    ds_negatives = pd.concat([ds_negatives_adjacent_subsegments, ds_negatives_downsample]).drop_duplicates().reset_index(drop=True)

    train_data, test_data = train_test_split(pd.concat([ds_positives, ds_negatives]), test_size=0.2, stratify=pd.concat([ds_positives, ds_negatives])['label'])
    
    return train_data, test_data

def tokenize_function(examples, tokenizer):
    return tokenizer(examples['patent_abstract'], padding="max_length", truncation=True, max_length=512)

def compute_metrics(eval_pred):
    metric = evaluate.combine(["accuracy", "recall", "precision", "f1"])
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

def train_model(subsegment, train_data, test_data, output_dir):
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
    
    train_dataset = Dataset.from_pandas(train_data).map(lambda x: tokenize_function(x, tokenizer), batched=True)
    test_dataset = Dataset.from_pandas(test_data).map(lambda x: tokenize_function(x, tokenizer), batched=True)
    
    training_args = TrainingArguments(
        output_dir=output_dir/f"model/SUBSEGMENT-{subsegment_name_processing(subsegment)}",
        num_train_epochs=5,
        per_device_train_batch_size=16,
        save_strategy='epoch',
        evaluation_strategy="epoch",
        load_best_model_at_end=True,
        logging_dir=output_dir/f"model/SUBSEGMENT-{subsegment_name_processing(subsegment)}/logs",
        logging_steps=100,
        log_level='error',
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics,
    )
    
    trainer.train()
    trainer.save_model(output_dir/f"model/SUBSEGMENT-{subsegment_name_processing(subsegment)}/best_model")
    
    # Evaluate on test set
    test_results = trainer.evaluate(eval_dataset=test_dataset)
    trainer.save_metrics('test', test_results)
    
    return trainer

def predict_all_models(df_predicting, ls_subsegment, output_dir):
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    
    # Preprocess the prediction data
    df_predicting['patent_abstract'] = df_predicting['patent_abstract'].fillna('')  # Replace NaN with empty string
    df_predicting['patent_abstract'] = df_predicting['patent_abstract'].astype(str)  # Ensure all entries are strings
    df_predicting = df_predicting[df_predicting['patent_abstract'].str.strip() != '']  # Remove empty strings
    
    predict_dataset = Dataset.from_pandas(df_predicting)
    
    def safe_tokenize_function(examples):
        return tokenizer(examples['patent_abstract'], padding="max_length", truncation=True, max_length=512)
    
    predict_dataset = predict_dataset.map(safe_tokenize_function, batched=True, remove_columns=predict_dataset.column_names)
    
    for subsegment in ls_subsegment:
        model_path = output_dir/f"model/SUBSEGMENT-{subsegment_name_processing(subsegment)}/best_model"
        if not model_path.exists():
            print(f"Model for {subsegment} not found. Skipping.")
            continue
        
        model = AutoModelForSequenceClassification.from_pretrained(model_path)
        trainer = Trainer(model=model)
        
        try:
            predictions = trainer.predict(predict_dataset)
            y_pred = predictions.predictions
            y_neg = predictions.predictions[:, 0]
            y_pos = predictions.predictions[:, 1]
            y_pred_dummy = np.argmax(y_pred, axis=-1)
            
            df_results = pd.DataFrame({
                'patent_id': df_predicting['patent_id'],
                'patent_abstract': df_predicting['patent_abstract'],
                'pred_pos': y_pos,
                'pred_neg': y_neg,
                'positive': y_pred_dummy
            })
            
            df_results.to_csv(output_dir/f"positive/{subsegment_name_processing(subsegment)}_patent_positive_bert.csv", index=False)
        except Exception as e:
            print(f"Error predicting for subsegment {subsegment}: {str(e)}")
            continue

def update_progress(progress, subsegment, status, output_dir, slice_id):
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    if subsegment not in progress:
        progress[subsegment] = {"status": status, "start_time": now, "last_update": now}
    else:
        progress[subsegment]["status"] = status
        progress[subsegment]["last_update"] = now
    
    progress_file = output_dir/f"progress_{slice_id}.json"
    with open(progress_file, 'w') as f:
        json.dump(progress, f, indent=2)

def main(start_index=0, end_index=None, slice_id="full"):
    output_dir = Path('output')
    output_dir.mkdir(exist_ok=True)
    (output_dir/"model").mkdir(exist_ok=True)
    (output_dir/"positive").mkdir(exist_ok=True)
    
    df_training, df_predicting, ls_subsegment = preprocess_data()
    
    # Slice the ls_subsegment if end_index is provided
    if end_index is not None:
        ls_subsegment = ls_subsegment[start_index:end_index]
    else:
        ls_subsegment = ls_subsegment[start_index:]
    
    progress = {}
    progress_file = output_dir/f"progress_{slice_id}.json"
    if progress_file.exists():
        with open(progress_file, 'r') as f:
            progress = json.load(f)
    
    for subsegment in ls_subsegment:
        if subsegment in progress and progress[subsegment]["status"] == "completed":
            print(f"Skipping {subsegment}, already completed.")
            continue
        
        print(f"Processing {subsegment}")
        update_progress(progress, subsegment, "in_progress", output_dir, slice_id)
        
        try:
            train_data, test_data = prepare_data_for_subsegment(df_training, subsegment)
            if train_data is not None and test_data is not None:
                trainer = train_model(subsegment, train_data, test_data, output_dir)
                update_progress(progress, subsegment, "completed", output_dir, slice_id)
            else:
                update_progress(progress, subsegment, "skipped", output_dir, slice_id)
        except Exception as e:
            print(f"Error processing {subsegment}: {str(e)}")
            update_progress(progress, subsegment, "error", output_dir, slice_id)
    
    print("All models in this slice trained. Starting predictions.")
    update_progress(progress, "prediction", "in_progress", output_dir, slice_id)
    try:
        predict_all_models(df_predicting, ls_subsegment, output_dir)
        update_progress(progress, "prediction", "completed", output_dir, slice_id)
    except Exception as e:
        print(f"Error during prediction: {str(e)}")
        update_progress(progress, "prediction", "error", output_dir, slice_id)
    
    print(f"Processing completed for slice {slice_id}")

if __name__ == "__main__":
    # Example usage:
    # To process the first 60 items:
    main(start_index=0, end_index=50, slice_id="slice1")